# CHMM toxicity generation

This notebook only generates continuations. It uses the sparse `src/chmm.py` implementation directly: clone states deterministically emit their token, transition mass is propagated through sparse token-pair blocks, and clone-level predictions are grouped back to GPT-2 token ids for TRACE-style guidance.


In [1]:
from __future__ import annotations

import csv
import json
import math
import os
import random
import sys
from pathlib import Path
from typing import Dict, List, Tuple

os.environ.setdefault("MKL_SERVICE_FORCE_INTEL", "1")
os.environ.setdefault("MKL_THREADING_LAYER", "GNU")

# modify this
PROJECT_ROOT = Path("/home/s/sukumarg/uncertainty/Ctrl-G/trace/trace").resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessorList

# modify this
from src.chmm import CHMM
from src.chmm_logits_processor import CHMMGuidedLogitsProcessor

torch.set_float32_matmul_precision("high")
print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


Project root: /home/s/sukumarg/uncertainty/Ctrl-G/trace/trace
PyTorch: 2.5.1+cu121
CUDA available: True


In [2]:
BASE_MODEL = "gpt2-large"

GUIDANCE_A = 1.0
MAX_NEW_TOKENS = 20
NUM_GENERATIONS = 3
GENERATION_BATCH_SIZE = 3
PROMPT_BATCH_SIZE = 32
INCLUDE_BASELINE = False
LIMIT_PROMPTS = None
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OVERWRITE_OUTPUT = True

# chmm_gpt2-large_uniform8
# chmm_gpt2-large_lognormal_20to2

# modify these
CHMM_MODEL_PATH = Path(
    "/home/s/sukumarg/uncertainty/Ctrl-G/distillation/workspace/models/chmm_gpt2-large_uniform8/checkpoint-200"
)
PROMPTS_PATH = PROJECT_ROOT / "real-toxicity-prompts/prompts.jsonl"
WEIGHTS_PATH = PROJECT_ROOT / "data/coefficients.csv"
OUTPUT_CSV = PROJECT_ROOT / "results" / f"chmm_toxicity_a{GUIDANCE_A}_generated.csv"

print(f"Base LM: {BASE_MODEL}")
print(f"CHMM path: {CHMM_MODEL_PATH}")
print(f"Prompts: {PROMPTS_PATH}")
print(f"Weights: {WEIGHTS_PATH}")
print(f"Output: {OUTPUT_CSV}")
print(f"Device: {DEVICE}")
if not (CHMM_MODEL_PATH / "model.pt").exists():
    print("Set CHMM_MODEL_PATH above to a trained src/chmm.py checkpoint directory containing model.pt.")


Base LM: gpt2-large
CHMM path: /home/s/sukumarg/uncertainty/Ctrl-G/distillation/workspace/models/chmm_gpt2-large_uniform8/checkpoint-200
Prompts: /home/s/sukumarg/uncertainty/Ctrl-G/trace/trace/real-toxicity-prompts/prompts.jsonl
Weights: /home/s/sukumarg/uncertainty/Ctrl-G/trace/trace/data/coefficients.csv
Output: /home/s/sukumarg/uncertainty/Ctrl-G/trace/trace/results/chmm_toxicity_a1.0_generated.csv
Device: cuda


In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_prompt_records(path: Path, limit: int | None = None) -> List[Tuple[int, str]]:
    records = []
    with path.open("r", encoding="utf-8") as fin:
        for idx, line in enumerate(fin):
            if limit is not None and len(records) >= limit:
                break
            item = json.loads(line)
            records.append((idx, item["prompt"]["text"]))
    return records


def load_coefficients_csv(path: Path, vocab_size: int, device: str | torch.device) -> torch.Tensor:
    weights = torch.zeros(vocab_size, dtype=torch.float32)
    seen = set()
    with path.open("r", encoding="utf-8", newline="") as fin:
        reader = csv.DictReader(fin)
        required = {"Token ID", "Coefficient"}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError(f"{path} must contain columns {sorted(required)}")
        for row in reader:
            token_id = int(row["Token ID"])
            if 0 <= token_id < vocab_size:
                weights[token_id] = float(row["Coefficient"])
                seen.add(token_id)
    if len(seen) != vocab_size:
        print(f"Warning: loaded coefficients for {len(seen)} / {vocab_size} tokens; missing tokens use 0.0.")
    return weights.to(device)


In [4]:
set_seed(SEED)

if not PROMPTS_PATH.exists():
    raise FileNotFoundError(f"Missing prompts file: {PROMPTS_PATH}")
if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(f"Missing weights file: {WEIGHTS_PATH}")
if not (CHMM_MODEL_PATH / "model.pt").exists():
    raise FileNotFoundError(f"Missing CHMM checkpoint model.pt in: {CHMM_MODEL_PATH}")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side="left")
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

print("Loading sparse CHMM and precomputing guidance cache...")
chmm = CHMM.from_pretrained(CHMM_MODEL_PATH, map_location=DEVICE).to(DEVICE).eval()
if tokenizer.vocab_size != chmm.vocab_size:
    raise ValueError(f"Tokenizer vocab size {tokenizer.vocab_size} != CHMM vocab size {chmm.vocab_size}")

coefficients = load_coefficients_csv(WEIGHTS_PATH, chmm.vocab_size, DEVICE)
processor = CHMMGuidedLogitsProcessor(
    chmm=chmm,
    coefficients=coefficients,
    horizon=MAX_NEW_TOKENS,
    a=GUIDANCE_A,
    device=DEVICE,
)

print("Loading language model...")
lm = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(DEVICE).eval()

prompts = load_prompt_records(PROMPTS_PATH, LIMIT_PROMPTS)
print(f"Loaded {len(prompts)} prompts")
print(f"CHMM hidden states: {chmm.hidden_states:,}")
print(f"CHMM sparse token-pair blocks: {int(chmm.pair_codes.numel()):,}")


Loading tokenizer...


Loading sparse CHMM and precomputing guidance cache...
Loading language model...


Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

Loaded 99442 prompts
CHMM hidden states: 402,056
CHMM sparse token-pair blocks: 15,291,569


In [5]:
def tokenize_prompts(prompt_texts: List[str], max_prompt_len: int):
    tokenizer_kwargs = dict(
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_prompt_len,
    )
    try:
        return tokenizer(prompt_texts, **tokenizer_kwargs)
    except (AttributeError, TypeError):
        encoded_rows = [
            tokenizer.encode_plus(text, **tokenizer_kwargs)
            for text in prompt_texts
        ]
        input_ids = torch.cat([row["input_ids"] for row in encoded_rows], dim=0)
        attention_mask = torch.cat([row["attention_mask"] for row in encoded_rows], dim=0)
        return {"input_ids": input_ids, "attention_mask": attention_mask}


def decode_continuations(generated: torch.Tensor, prompt_len: int, batch_size: int, num_return_sequences: int) -> Dict[int, List[str]]:
    out = {batch_idx: [] for batch_idx in range(batch_size)}
    for batch_idx in range(batch_size):
        for sample_idx in range(num_return_sequences):
            seq_idx = batch_idx * num_return_sequences + sample_idx
            continuation_ids = generated[seq_idx, prompt_len:]
            text = tokenizer.decode(
                continuation_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )
            out[batch_idx].append(text)
    return out


def generate_batch(prompt_texts: List[str], use_chmm: bool) -> Dict[int, List[str]]:
    max_context = getattr(lm.config, "max_position_embeddings", None) or getattr(lm.config, "n_positions", 1024)
    max_prompt_len = max(int(max_context) - MAX_NEW_TOKENS - 10, 10)
    inputs = tokenize_prompts(prompt_texts, max_prompt_len)
    prompt_ids = inputs["input_ids"].to(DEVICE)
    attention_mask = inputs["attention_mask"].to(DEVICE)
    prompt_len = prompt_ids.shape[1]

    continuations = {idx: [] for idx in range(len(prompt_texts))}
    loops = math.ceil(NUM_GENERATIONS / GENERATION_BATCH_SIZE)

    for loop_idx in range(loops):
        num_to_generate = min(
            GENERATION_BATCH_SIZE,
            NUM_GENERATIONS - loop_idx * GENERATION_BATCH_SIZE,
        )
        if num_to_generate <= 0:
            break

        logits_processors = LogitsProcessorList([])
        if use_chmm:
            processor.configure_for_prompts(
                prompt_ids,
                attention_mask,
                repeat_interleave=num_to_generate,
            )
            logits_processors = LogitsProcessorList([processor])

        with torch.no_grad():
            generated = lm.generate(
                input_ids=prompt_ids,
                attention_mask=attention_mask,
                logits_processor=logits_processors,
                max_new_tokens=MAX_NEW_TOKENS,
                num_return_sequences=num_to_generate,
                do_sample=True,
                top_p=0.9,
                top_k=0,
                temperature=1.0,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        loop_outputs = decode_continuations(generated, prompt_len, len(prompt_texts), num_to_generate)
        for batch_idx, texts in loop_outputs.items():
            continuations[batch_idx].extend(texts)

    return continuations


In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
if OUTPUT_CSV.exists() and not OVERWRITE_OUTPUT:
    raise FileExistsError(f"Output exists and OVERWRITE_OUTPUT=False: {OUTPUT_CSV}")

fieldnames = ["index", "prefix"]
fieldnames.extend(f"trace_gen_{idx + 1}" for idx in range(NUM_GENERATIONS))
if INCLUDE_BASELINE:
    fieldnames.extend(f"baseline_gen_{idx + 1}" for idx in range(NUM_GENERATIONS))

with OUTPUT_CSV.open("w", encoding="utf-8", newline="") as fout:
    writer = csv.DictWriter(fout, fieldnames=fieldnames)
    writer.writeheader()

    for start in tqdm(range(0, len(prompts), PROMPT_BATCH_SIZE), desc="Generating"):
        batch = prompts[start : start + PROMPT_BATCH_SIZE]
        prompt_texts = [text for _, text in batch]

        chmm_outputs = generate_batch(prompt_texts, use_chmm=True)
        baseline_outputs = generate_batch(prompt_texts, use_chmm=False) if INCLUDE_BASELINE else {}

        for batch_idx, (orig_idx, prompt_text) in enumerate(batch):
            row = {"index": orig_idx, "prefix": prompt_text}
            for gen_idx, continuation in enumerate(chmm_outputs[batch_idx][:NUM_GENERATIONS]):
                row[f"trace_gen_{gen_idx + 1}"] = json.dumps({"continuation": continuation})
            if INCLUDE_BASELINE:
                for gen_idx, continuation in enumerate(baseline_outputs[batch_idx][:NUM_GENERATIONS]):
                    row[f"baseline_gen_{gen_idx + 1}"] = json.dumps({"continuation": continuation})
            writer.writerow(row)
            fout.flush()

print(f"Saved generated continuations to {OUTPUT_CSV}")
if processor.zero_probability_resets:
    print(f"Warning: reset {processor.zero_probability_resets} zero-probability CHMM posterior rows during generation.")


Generating:   0%|          | 0/3108 [00:00<?, ?it/s]

In [ ]:
with OUTPUT_CSV.open("r", encoding="utf-8", newline="") as fin:
    reader = csv.DictReader(fin)
    for row_idx, row in enumerate(reader):
        print(f"Prompt {row['index']}: {row['prefix']}")
        for col in [name for name in reader.fieldnames or [] if name.startswith('trace_gen_')][:2]:
            print(f"  {col}: {json.loads(row[col])['continuation']!r}")
        if row_idx >= 2:
            break
